In [1]:
import os
import gc
import time
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# ── Constants ────────────────────────────────────────────────
RANDOM_STATE  = 42
PARQUET_DIR   = "/nvme1/users/md962/glacier/Glacier Project/Merged/"
MODELS_DIR    = "/nvme1/users/md962/glacier/Glacier Project/Final_Models/"
RESULTS_DIR   = "/nvme1/users/md962/glacier/Glacier Project/Results/"
TENSORS_DIR   = "/nvme1/users/md962/glacier/Glacier Project/Tensors/"
PREDS_DIR     = "/nvme1/users/md962/glacier/Glacier Project/Predictions/"
TRAIN_REGIONS = ['r1a', 'r1b', 'r2']
VAL_REGION    = 'r3'
AE_COLS       = [f'A{i:02d}' for i in range(64)]
EASD_COLS     = ['elevation', 'edge_distance', 'aspect', 'slope']
RF_SAMPLE_N   = 2_000_000

for d in [MODELS_DIR, RESULTS_DIR, TENSORS_DIR, PREDS_DIR]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition


In [2]:
# Load r3 first so its pixels take priority in dedup
dfs = []
for region in ['r3', 'r1a', 'r1b', 'r2']:  # r3 first
    df = pd.read_parquet(f"{PARQUET_DIR}{region}_combined.parquet")
    df['region'] = region
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all = df_all.drop_duplicates(subset=['lon', 'lat'], keep='first').reset_index(drop=True)

print(f"Total unique pixels: {len(df_all):,}")
for region in ['r1a', 'r1b', 'r2', 'r3']:
    df_r = df_all[df_all['region'] == region]
    print(f"  {region}: {len(df_r):,} pixels, melt rate: {df_r['melt_label'].mean():.3f}")

Total unique pixels: 13,996,828
  r1a: 6,222,913 pixels, melt rate: 0.139
  r1b: 1,028,089 pixels, melt rate: 0.264
  r2: 4,971,507 pixels, melt rate: 0.239
  r3: 1,774,319 pixels, melt rate: 0.240


In [3]:
df_train = df_all[df_all['region'].isin(['r1a', 'r1b', 'r2'])].reset_index(drop=True)
df_val   = df_all[df_all['region'] == 'r3'].reset_index(drop=True)

print(f"Train pixels: {len(df_train):,}, melt rate: {df_train['melt_label'].mean():.3f}")
print(f"Val pixels:   {len(df_val):,},   melt rate: {df_val['melt_label'].mean():.3f}")

Train pixels: 12,222,509, melt rate: 0.190
Val pixels:   1,774,319,   melt rate: 0.240


In [7]:
ROUND_DP = 6

def build_neighbour_table(df, patch_size=3, tolerance=0.6):
    half   = patch_size // 2
    coords = np.stack([df['lon'].values, df['lat'].values], axis=1)
    pixel_lon = np.median(np.diff(np.sort(df['lon'].unique())))
    pixel_lat = np.median(np.diff(np.sort(df['lat'].unique())))
    print(f"  Building KD-tree for {len(df):,} pixels...")
    tree = cKDTree(coords)
    k = patch_size ** 2 + 1
    distances, indices = tree.query(coords, k=k, workers=-1)
    neighbour_table = np.full((len(df), patch_size, patch_size), -1, dtype=np.int32)
    
    for slot in range(k):
        neighbour_coords = coords[indices[:, slot]]
        delta_lon = neighbour_coords[:, 0] - coords[:, 0]
        delta_lat = neighbour_coords[:, 1] - coords[:, 1]
        dc = np.round(delta_lon / pixel_lon).astype(int)
        dr = np.round(delta_lat / pixel_lat).astype(int)
        
    # Always compute patch_row/col before validity check
        patch_row = (dr + half).clip(0, patch_size - 1)
        patch_col = (dc + half).clip(0, patch_size - 1)
        
        residual_lon = np.abs(delta_lon - dc * pixel_lon)
        residual_lat = np.abs(delta_lat - dr * pixel_lat)
        
        valid = (
            (np.abs(dc) <= half) & (np.abs(dr) <= half) &
            (residual_lon < tolerance * pixel_lon) &
            (residual_lat < tolerance * pixel_lat) &
            (patch_row >= 0) & (patch_row < patch_size) &
            (patch_col >= 0) & (patch_col < patch_size)
        )
        
        pixel_indices = np.arange(len(df))
        mask = valid & (neighbour_table[pixel_indices, patch_row, patch_col] == -1)
        neighbour_table[pixel_indices[mask], patch_row[mask], patch_col[mask]] = \
            indices[mask, slot]
    
    return neighbour_table, pixel_lon, pixel_lat


def build_gpu_tensors_pixel(df):
    """Single pixel — no neighbourhood. Input shape: (N, 64)"""
    embeddings = df[AE_COLS].values.astype(np.float32)   # (N, 64)
    labels     = df['melt_label'].values.astype(np.float32)
    print(f"  Moving {len(df):,} pixels to GPU...")
    return TensorDataset(
        torch.from_numpy(embeddings).cuda(),
        torch.from_numpy(labels).cuda()
    )

def build_gpu_tensors_patch(df, patch_size=3, device='cuda'):
    nt, _, _ = build_neighbour_table(df, patch_size)
    n    = len(df)
    flat = nt.reshape(n, patch_size * patch_size)
    missing   = flat == -1
    flat_safe = flat.copy()
    flat_safe[missing] = 0
    embeddings = df[AE_COLS].values.astype(np.float32)
    patches    = embeddings[flat_safe]
    patches[missing] = 0.0
    patches = patches.transpose(0, 2, 1).reshape(n, 64, patch_size, patch_size)
    labels  = df['melt_label'].values.astype(np.float32)
    print(f"  Moving {n:,} patches to {device}...")
    return TensorDataset(
        torch.from_numpy(patches).to(device),
        torch.from_numpy(labels).to(device)
    )

class GPUDataLoader:
    def __init__(self, tensor_dataset, batch_size, shuffle=True):
        self.patches    = tensor_dataset.tensors[0]
        self.labels     = tensor_dataset.tensors[1]
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.n          = len(self.labels)
        self.dataset    = self
    def __len__(self):
        return (self.n + self.batch_size - 1) // self.batch_size
    def __iter__(self):
        idx = torch.randperm(self.n, device='cuda') if self.shuffle \
              else torch.arange(self.n, device='cuda')
        for start in range(0, self.n, self.batch_size):
            batch_idx = idx[start:start + self.batch_size]
            yield self.patches[batch_idx], self.labels[batch_idx]

def run_epoch(model, loader, criterion, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    with torch.set_grad_enabled(train):
        for patches, labels in loader:
            logits = model(patches)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().detach().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / loader.n
    auc      = roc_auc_score(all_labels, all_probs)
    return avg_loss, auc

def train_model(model, train_loader, val_loader, pos_weight,
                n_epochs=20, patience=5, save_path=None):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )
    best_val_auc = 0
    best_epoch   = 0
    print(f"\n{'Epoch':>5}  {'Train Loss':>10}  {'Train AUC':>9}  "
          f"{'Val Loss':>8}  {'Val AUC':>7}")
    print("-" * 55)
    for epoch in range(1, n_epochs + 1):
        train_loss, train_auc = run_epoch(model, train_loader, criterion, optimizer, train=True)
        val_loss,   val_auc   = run_epoch(model, val_loader,   criterion, train=False)
        scheduler.step(val_auc)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch   = epoch
            if save_path:
                torch.save(model.state_dict(), save_path)
        print(f"{epoch:>5}  {train_loss:>10.4f}  {train_auc:>9.4f}  "
              f"{val_loss:>8.4f}  {val_auc:>7.4f}"
              + (" ← best" if epoch == best_epoch else ""))
        if epoch - best_epoch >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break
    print(f"\nBest val AUC: {best_val_auc:.4f} at epoch {best_epoch}")
    return best_val_auc

def predict_pytorch(model, tensor_dataset, batch_size=65536):
    """Run inference on a TensorDataset, return probability array."""
    model.eval()
    loader = GPUDataLoader(tensor_dataset, batch_size=batch_size, shuffle=False)
    probs  = []
    with torch.no_grad():
        for patches, _ in loader:
            logits = model(patches)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
    return np.array(probs, dtype=np.float32)


In [8]:
print("Building 5x5 patch tensors on CPU...")
all_patches_list = []
all_labels_list  = []

for region in TRAIN_REGIONS:
    print(f"  {region}...")
    df = df_train[df_train['region'] == region].reset_index(drop=True)
    tensors = build_gpu_tensors_patch(df, patch_size=5, device='cpu')
    all_patches_list.append(tensors.tensors[0])
    all_labels_list.append(tensors.tensors[1])
    del tensors, df
    gc.collect()

# Concatenate on CPU
all_patches = torch.cat(all_patches_list)
all_labels  = torch.cat(all_labels_list)
del all_patches_list, all_labels_list
gc.collect()
print(f"Total training patches (CPU): {len(all_labels):,}")
print(f"Melt rate: {all_labels.mean():.3f}")

# Move everything to GPU in one shot
print("Moving to GPU...")
all_patches = all_patches.to('cuda')
all_labels  = all_labels.to('cuda')
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Val tensors
print("Building val tensors...")
cnn5_val_tensors = build_gpu_tensors_patch(df_val, patch_size=5, device='cuda')

train_combined    = TensorDataset(all_patches, all_labels)
melt_rate         = all_labels.cpu().mean().item()
pos_weight        = torch.tensor((1 - melt_rate) / melt_rate)
cnn5_train_loader = GPUDataLoader(train_combined,   batch_size=65536, shuffle=True)
cnn5_val_loader   = GPUDataLoader(cnn5_val_tensors, batch_size=65536, shuffle=False)

print(f"Train loader: {len(cnn5_train_loader)} batches")
print(f"Val loader:   {len(cnn5_val_loader)} batches")
print(f"pos_weight:   {pos_weight:.2f}")

Building 5x5 patch tensors on CPU...
  r1a...
  Building KD-tree for 6,222,913 pixels...
  Moving 6,222,913 patches to cpu...
  r1b...
  Building KD-tree for 1,028,089 pixels...
  Moving 1,028,089 patches to cpu...
  r2...
  Building KD-tree for 4,971,507 pixels...
  Moving 4,971,507 patches to cpu...
Total training patches (CPU): 12,222,509
Melt rate: 0.190
Moving to GPU...
GPU memory: 78.3 GB
Building val tensors...
  Building KD-tree for 1,774,319 pixels...
  Moving 1,774,319 patches to cuda...
Train loader: 187 batches
Val loader:   28 batches
pos_weight:   4.26


In [9]:
class GlacierCNN5(nn.Module):
    """
    5x5 patch CNN.
    Input:  (B, 64, 5, 5)
    Output: (B,) raw logit
    """
    def __init__(self, dropout=0.5):
        super().__init__()
        self.conv_block = nn.Sequential(
            # (B, 64, 5, 5) -> (B, 128, 4, 4)
            nn.Conv2d(64, 128, kernel_size=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # (B, 128, 4, 4) -> (B, 256, 3, 3)
            nn.Conv2d(128, 256, kernel_size=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            # (B, 256, 3, 3) -> (B, 256, 1, 1)
            nn.Conv2d(256, 256, kernel_size=3),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        self.fc_block = nn.Sequential(
            nn.Flatten(),                    # (B, 256)
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.fc_block(self.conv_block(x)).squeeze(1)

In [10]:
cnn5_model = GlacierCNN5(dropout=0.5).to(device)
dummy = torch.zeros(8, 64, 5, 5).to(device)
print(f"CNN5 output shape: {cnn5_model(dummy).shape}")
print(f"CNN5 parameters: {sum(p.numel() for p in cnn5_model.parameters()):,}")

best_cnn5_auc = train_model(
    cnn5_model,
    cnn5_train_loader,
    cnn5_val_loader,
    pos_weight,
    n_epochs=20,
    patience=5,
    save_path=f"{MODELS_DIR}CNN5_PATCH5_R3HOLDOUT.pt"
)

CNN5 output shape: torch.Size([8])
CNN5 parameters: 792,641

Epoch  Train Loss  Train AUC  Val Loss  Val AUC
-------------------------------------------------------
    1      0.4129     0.9605    1.1663   0.9216 ← best
    2      0.3271     0.9734    1.6725   0.9257 ← best
    3      0.2978     0.9775    2.1289   0.9199
    4      0.2803     0.9798    2.4862   0.9225
    5      0.2653     0.9817    2.0840   0.9198
    6      0.2412     0.9845    2.2300   0.9135
    7      0.2328     0.9855    2.4544   0.9160

Early stopping at epoch 7

Best val AUC: 0.9257 at epoch 2


In [12]:
# Free training tensors from GPU
del all_patches, all_labels, train_combined
del cnn5_train_loader, cnn5_val_loader
del cnn5_val_tensors
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory after clearing: {torch.cuda.memory_allocated()/1e9:.1f} GB")

GPU memory after clearing: 0.1 GB


In [13]:
print("Running Peru-wide inference (CNN5 only)...")
all_probs_cnn5 = []
all_meta       = []

for region in ['r3', 'r1a', 'r1b', 'r2']:
    print(f"\n{region}...")
    df = df_all[df_all['region'] == region].reset_index(drop=True)
    
    cnn5_tensors = build_gpu_tensors_patch(df, patch_size=5, device='cuda')
    cnn5_p = predict_pytorch(cnn5_model, cnn5_tensors)
    del cnn5_tensors; torch.cuda.empty_cache()
    
    all_probs_cnn5.append(cnn5_p)
    all_meta.append(df[['lon', 'lat']].copy())
    
    print(f"  CNN5 mean prob: {cnn5_p.mean():.3f}")
    del df; gc.collect()

df_meta = pd.concat(all_meta, ignore_index=True)
df_meta['prob_cnn5_patch5'] = np.concatenate(all_probs_cnn5)

Running Peru-wide inference (CNN5 only)...

r3...
  Building KD-tree for 1,774,319 pixels...
  Moving 1,774,319 patches to cuda...
  CNN5 mean prob: 0.182

r1a...
  Building KD-tree for 6,222,913 pixels...
  Moving 6,222,913 patches to cuda...
  CNN5 mean prob: 0.187

r1b...
  Building KD-tree for 1,028,089 pixels...
  Moving 1,028,089 patches to cuda...
  CNN5 mean prob: 0.325

r2...
  Building KD-tree for 4,971,507 pixels...
  Moving 4,971,507 patches to cuda...
  CNN5 mean prob: 0.302


In [14]:
# Load existing predictions
df_preds = pd.read_parquet(f"{PREDS_DIR}predictions_full_peru_r3holdout.parquet")

# Merge CNN5 predictions
df_preds = df_preds.merge(
    df_meta[['lon', 'lat', 'prob_cnn5_patch5']],
    on=['lon', 'lat'],
    how='left'
)

print(f"CNN5 nulls: {df_preds['prob_cnn5_patch5'].isna().sum():,}")

df_preds.to_parquet(
    f"{PREDS_DIR}predictions_full_peru_r3holdout.parquet",
    index=False
)
print(f"Saved {len(df_preds):,} rows")

# Evaluate
MODELS = {
    'RF (EASD)':     'prob_rf_easd',
    'MLP (AE64)':    'prob_mlp_ae64',
    'CNN (patch3)':  'prob_cnn_patch3',
    'CNN (patch5)':  'prob_cnn5_patch5'
}

def overlap_table(df, title):
    actual = df['melt_label'].values == 1
    n_melt = actual.sum()
    print(f"\n=== {title} ===")
    print(f"Pixels: {len(df):,}, Melt: {n_melt:,} ({n_melt/len(df)*100:.1f}%)")
    print(f"{'Model':<20} {'Overlap %':>10} {'IoU':>8} {'AUC':>8}")
    print("-" * 50)
    for name, col in MODELS.items():
        probs = df[col].values
        top   = np.argsort(probs)[::-1][:n_melt]
        pred  = np.zeros(len(probs), dtype=bool)
        pred[top] = True
        n_int = (pred & actual).sum()
        ovlp  = n_int / n_melt * 100
        iou   = n_int / (pred | actual).sum()
        auc   = roc_auc_score(actual, probs)
        print(f"{name:<20} {ovlp:>9.2f}% {iou:>8.4f} {auc:>8.4f}")

        
overlap_table(df_preds, "PERU-WIDE (includes training regions)")
overlap_table(df_preds[df_preds['region'] == 'r3'], "R3 ONLY (out-of-sample for all models)")

CNN5 nulls: 0
Saved 13,996,828 rows

=== PERU-WIDE (includes training regions) ===
Pixels: 13,996,828, Melt: 2,751,035 (19.7%)
Model                 Overlap %      IoU      AUC
--------------------------------------------------
RF (EASD)                77.82%   0.6369   0.9528
MLP (AE64)               81.10%   0.6821   0.9698
CNN (patch3)             84.54%   0.7322   0.9773
CNN (patch5)             84.42%   0.7303   0.9717

=== R3 ONLY (out-of-sample for all models) ===
Pixels: 1,774,319, Melt: 426,115 (24.0%)
Model                 Overlap %      IoU      AUC
--------------------------------------------------
RF (EASD)                70.29%   0.5418   0.8987
MLP (AE64)               79.52%   0.6600   0.9454
CNN (patch3)             79.85%   0.6646   0.9451
CNN (patch5)             73.37%   0.5795   0.9160
